# ZFNet vs CvT: US cars (196 classes) -> Pakistani cars (48 classes)

**Pipeline**
0. **Splits:** for each dataset, `train/` and `test/` are merged and re-split 70/10/20 (train/val/test), stratified per class.
1. **Stage 1 (source domain):** train each model on the US train split (196 classes), evaluate on the US test split.
2. **Stage 2 (target domain):** keep the backbone, replace the classifier with a new 48-way head, train on the Pakistani train split, and evaluate on the Pakistani test split.

For Stage 2 we compare three settings per model:
- `baseline`: no US-car training (ZFNet from scratch, CvT from ImageNet weights only)
- `us_finetune`: US-car backbone, full fine-tuning with a small backbone LR
- `us_linear`: US-car backbone frozen, only the new head is trained (linear probe)

**Before running:** Runtime > Change runtime type > GPU. Then set the two paths in the config cell.
Set `FAST = True` first to smoke-test the whole notebook in a few minutes, then set it back to `False`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn", "seaborn"], check=False)

import os, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive")
else:
    BASE_DIR = Path(".")

# CONFIG 
US_ZIP  = BASE_DIR / "car_data_us.zip"
PAK_ZIP = BASE_DIR / "car_data_pak.zip"
OUT_DIR = BASE_DIR / "cnn_assignment_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)



import zipfile
def prepare_dataset(zip_path):
    """Unzip to Colab's fast local disk (once) and return the folder that holds train/ and test/."""
    dest = Path("./data") / zip_path.stem
    if not (dest.exists() and any(dest.iterdir())):
        assert zip_path.exists(), f"{zip_path} not found - check the file name and that Drive is mounted"
        print(f"extracting {zip_path.name} ... (a few minutes the first time)")
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
    for d, subdirs, _ in os.walk(dest):
        subdirs[:] = [s for s in subdirs if s != "__MACOSX"]
        if "train" in subdirs and "test" in subdirs:
            return Path(d)
    raise FileNotFoundError(f"no folder containing train/ and test/ found inside {zip_path.name}")

US_ROOT  = prepare_dataset(US_ZIP)
PAK_ROOT = prepare_dataset(PAK_ZIP)
print("US data :", US_ROOT)
print("PAK data:", PAK_ROOT)
OUT_DIR  = Path("/content/drive/MyDrive/cnn_assignment_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FAST       = False   # True = tiny subsets and 1 epoch, only to check that the code runs
RESUME     = True    # skip runs already recorded in results.csv (ignored when FAST = True)
SEED       = 42
IMG        = 224
BATCH      = 64
WORKERS    = 2
VAL_FRAC   = 0.10    # share of ALL images used for validation / early stopping
TEST_FRAC  = 0.20    # share of ALL images used for final testing (train gets the remaining 70%)
EXPECT_US, EXPECT_PAK = 196, 48

# Per-model hyperparameters
HP = {
    "zfnet": dict(opt="sgd",   us_ep=60, pak_ep=40,
                  us_lr=(0.01, 0.01),   ft_lr=(1e-3, 1e-2),  lin_lr=1e-2, base_lr=(0.01, 0.01)),
    "cvt":   dict(opt="adamw", us_ep=15, pak_ep=15,
                  us_lr=(5e-5, 1e-3),   ft_lr=(2e-5, 5e-4),  lin_lr=1e-3, base_lr=(5e-5, 1e-3)),
}
PATIENCE = 8
# ------------------------------------------------------------------

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

## 1. Data
Each root has `train/<class>/*.jpg` and `test/<class>/*.jpg`. We **merge both folders** and re-split every dataset
into a stratified **70% train / 10% validation / 20% test** (same seed, so the split is identical on every run).
Every class appears in all three splits. The test split is used only for final evaluation.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

def make_loaders(root, expect_classes):
    root = Path(root)
    # Merge the provided train/ and test/ folders, then re-split ourselves.
    # Two views of the same files: augmented (for training) and clean (for val/test).
    parts_aug  = [datasets.ImageFolder(root / s, train_tf) for s in ("train", "test")]
    parts_eval = [datasets.ImageFolder(root / s, eval_tf)  for s in ("train", "test")]
    classes = parts_aug[0].classes
    assert len(classes) == expect_classes, f"{root}: found {len(classes)} classes"
    assert parts_aug[1].classes == classes, "train/ and test/ class folders differ"
    full_aug, full_eval = ConcatDataset(parts_aug), ConcatDataset(parts_eval)
    targets = np.array(parts_aug[0].targets + parts_aug[1].targets)

    idx = np.arange(len(targets))
    trval_idx, test_idx = train_test_split(idx, test_size=TEST_FRAC, stratify=targets, random_state=SEED)
    tr_idx, va_idx = train_test_split(trval_idx, test_size=VAL_FRAC / (1 - TEST_FRAC),
                                      stratify=targets[trval_idx], random_state=SEED)
    if FAST:
        tr_idx, va_idx, test_idx = tr_idx[:300], va_idx[:100], test_idx[:200]

    kw = dict(batch_size=BATCH, num_workers=WORKERS, pin_memory=USE_AMP, persistent_workers=WORKERS > 0)
    loaders = dict(
        train=DataLoader(Subset(full_aug, tr_idx), shuffle=True, drop_last=True, **kw),
        val=DataLoader(Subset(full_eval, va_idx), shuffle=False, **kw),
        test=DataLoader(Subset(full_eval, test_idx), shuffle=False, **kw),
    )
    print(f"{root.name}: classes={len(classes)} total={len(idx)} -> train={len(tr_idx)} val={len(va_idx)} test={len(test_idx)}")
    return loaders, classes

US_LOADERS,  US_CLASSES  = make_loaders(US_ROOT,  EXPECT_US)
PAK_LOADERS, PAK_CLASSES = make_loaders(PAK_ROOT, EXPECT_PAK)

## 2. Models
Both models expose the same two methods: `head()` returns the final classification layer and `reset_head(n)` swaps it for a new `n`-way layer.

- **ZFNet**: AlexNet-style network with 7x7 stride-2 first conv (96 filters), then 5x5 (256), 3x3 (384, 384, 256) and three FC layers. We add BatchNorm after each conv so it trains stably from scratch (the original used local response normalisation).
- **CvT-13**: HuggingFace `microsoft/cvt-13` with ImageNet weights (convolutional token embedding, convolutional projection in attention, 3 stages).

In [ ]:
class ZFNet(nn.Module):
    def __init__(self, num_classes, bn=True):
        super().__init__()
        def conv(i, o, k, s=1, p=0):
            layers = [nn.Conv2d(i, o, k, s, p, bias=not bn)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.ReLU(inplace=True))
            return layers
        self.features = nn.Sequential(
            *conv(3, 96, 7, 2, 1),  nn.MaxPool2d(3, 2),
            *conv(96, 256, 5, 2),   nn.MaxPool2d(3, 2),
            *conv(256, 384, 3, 1, 1),
            *conv(384, 384, 3, 1, 1),
            *conv(384, 256, 3, 1, 1), nn.MaxPool2d(3, 2),
        )
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )
    def forward(self, x):
        return self.classifier(torch.flatten(self.pool(self.features(x)), 1))
    def head(self): return self.classifier[-1]
    def reset_head(self, n): self.classifier[-1] = nn.Linear(4096, n)


class CvT(nn.Module):
    NAME = "microsoft/cvt-13"
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        from transformers import CvtForImageClassification, CvtConfig
        if pretrained:
            self.m = CvtForImageClassification.from_pretrained(
                self.NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
        else:
            self.m = CvtForImageClassification(CvtConfig.from_pretrained(self.NAME, num_labels=num_classes))
    def forward(self, x): return self.m(pixel_values=x).logits
    def head(self): return self.m.classifier
    def reset_head(self, n): self.m.classifier = nn.Linear(self.m.classifier.in_features, n)


def build(name, num_classes, pretrained=True):
    if name == "zfnet": return ZFNet(num_classes).to(DEVICE)
    if name == "cvt":   return CvT(num_classes, pretrained).to(DEVICE)
    raise ValueError(name)

def count_params(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only))

# parameter counts for the report
rows = []
for n in ["zfnet", "cvt"]:
    for k in (EXPECT_US, EXPECT_PAK):
        m = build(n, k); rows.append(dict(model=n, num_classes=k, params_millions=round(count_params(m) / 1e6, 2)))
        del m
print(pd.DataFrame(rows).to_string(index=False))
with torch.no_grad():  # shape sanity check
    for n in ["zfnet", "cvt"]:
        m = build(n, EXPECT_US).eval(); print(n, "output:", tuple(m(torch.randn(2, 3, IMG, IMG, device=DEVICE)).shape)); del m

## 3. Training and evaluation utilities

In [ ]:
def run_epoch(model, loader, criterion, opt=None, scaler=None, linear_probe=False):
    train = opt is not None
    if train:
        model.train()
        if linear_probe:             # frozen backbone: keep BatchNorm/Dropout in eval mode
            model.eval(); model.head().train()
        params = [p for p in model.parameters() if p.requires_grad]
    else:
        model.eval()
    tot_loss = tot_correct = tot = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 5.0)
                scaler.step(opt); scaler.update()
            tot_loss += loss.item() * x.size(0)
            tot_correct += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    return tot_loss / tot, tot_correct / tot


def make_optimizer(model, opt_name, lr_backbone, lr_head, linear_probe):
    head_ids = {id(p) for p in model.head().parameters()}
    for p in model.parameters():
        p.requires_grad = (id(p) in head_ids) or not linear_probe
    head = [p for p in model.parameters() if id(p) in head_ids]
    back = [p for p in model.parameters() if id(p) not in head_ids and p.requires_grad]
    groups = [dict(params=head, lr=lr_head)] + ([dict(params=back, lr=lr_backbone)] if back else [])
    if opt_name == "sgd":
        return torch.optim.SGD(groups, momentum=0.9, weight_decay=5e-4, nesterov=True)
    return torch.optim.AdamW(groups, weight_decay=0.05)


def fit(model, loaders, epochs, opt_name, lr_backbone, lr_head, linear_probe=False, tag=""):
    epochs = 1 if FAST else epochs
    opt = make_optimizer(model, opt_name, lr_backbone, lr_head, linear_probe)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist, best_acc, best_state, bad = [], -1, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        trl, tra = run_epoch(model, loaders["train"], crit, opt, scaler, linear_probe)
        val, vaa = run_epoch(model, loaders["val"], crit)
        sched.step()
        hist.append(dict(epoch=ep, train_loss=trl, train_acc=tra, val_loss=val, val_acc=vaa))
        print(f"[{tag}] ep {ep:02d}/{epochs}  train {trl:.3f}/{tra:.3f}  val {val:.3f}/{vaa:.3f}")
        if vaa > best_acc:
            best_acc, bad = vaa, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: print(f"[{tag}] early stop"); break
    model.load_state_dict(best_state)
    return pd.DataFrame(hist), (time.time() - t0) / 60


@torch.no_grad()
def evaluate(model, loader, num_classes):
    model.eval(); preds, tgts, top5 = [], [], 0
    k = min(5, num_classes)
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(x)
        top5 += (out.topk(k, 1).indices.cpu() == y[:, None]).any(1).sum().item()
        preds.append(out.argmax(1).cpu()); tgts.append(y)
    p, t = torch.cat(preds).numpy(), torch.cat(tgts).numpy()
    return dict(top1=(p == t).mean(), top5=top5 / len(t),
                macro_f1=f1_score(t, p, average="macro", zero_division=0), preds=p, targets=t)

RESULTS, HISTORIES, PREDS = [], {}, {}

def log_result(model_name, stage, setting, dataset, res, model, minutes):
    RESULTS.append(dict(model=model_name, stage=stage, setting=setting, dataset=dataset,
                        top1=round(res["top1"] * 100, 2), top5=round(res["top5"] * 100, 2),
                        macro_f1=round(res["macro_f1"] * 100, 2),
                        params_M=round(count_params(model) / 1e6, 2),
                        trainable_M=round(count_params(model, True) / 1e6, 2),
                        train_min=round(minutes, 1)))
    print(RESULTS[-1])
    pd.DataFrame(RESULTS).to_csv(OUT_DIR / "results.csv", index=False)

## 4. Stage 1: train on the US dataset (196 classes), test on US test split
ZFNet starts from scratch (no official ImageNet weights exist for it). CvT starts from ImageNet weights.

In [ ]:
# Resume support: reload earlier results so that finished runs are skipped
RESULTS_CSV = OUT_DIR / "results.csv"
if RESUME and not FAST and RESULTS_CSV.exists() and not RESULTS:
    RESULTS.extend(pd.read_csv(RESULTS_CSV).to_dict("records"))
DONE = {(r["model"], r["stage"], r["setting"]) for r in RESULTS}
print("already done:", sorted(DONE))

def load_saved_hist(key, path):
    if path.exists(): HISTORIES[key] = pd.read_csv(path)

CKPT = {}
for name in ["zfnet", "cvt"]:
    CKPT[name] = OUT_DIR / f"{name}_us.pt"
    if (name, "stage1", "US train") in DONE and CKPT[name].exists():
        print(f"[{name}-US] already trained, skipping (using {CKPT[name].name})")
        load_saved_hist(f"{name}/stage1", OUT_DIR / f"hist_{name}_stage1.csv")
        continue
    set_seed()
    model = build(name, EXPECT_US)
    hp = HP[name]
    hist, mins = fit(model, US_LOADERS, hp["us_ep"], hp["opt"], *hp["us_lr"], tag=f"{name}-US")
    res = evaluate(model, US_LOADERS["test"], EXPECT_US)
    log_result(name, "stage1", "US train", "US test", res, model, mins)
    HISTORIES[f"{name}/stage1"] = hist
    hist.to_csv(OUT_DIR / f"hist_{name}_stage1.csv", index=False)
    torch.save(model.state_dict(), CKPT[name])
    del model; torch.cuda.empty_cache()

## 5. Stage 2: adapt to the Pakistani dataset (48 classes)
`baseline` skips the US training, so comparing it against `us_finetune` and `us_linear` shows whether US car data helps.

In [ ]:
def run_pak(name, mode):
    set_seed()
    hp = HP[name]
    if mode == "baseline":
        model = build(name, EXPECT_PAK)                       # fresh 48-way model
        lr_b, lr_h = hp["base_lr"]; lin = False
    else:
        model = build(name, EXPECT_US, pretrained=False)      # architecture only
        model.load_state_dict(torch.load(CKPT[name], map_location=DEVICE))
        model.reset_head(EXPECT_PAK); model.to(DEVICE)        # new 48-way head
        if mode == "us_finetune": (lr_b, lr_h), lin = hp["ft_lr"], False
        else:                     lr_b, lr_h, lin = 0.0, hp["lin_lr"], True
    hist, mins = fit(model, PAK_LOADERS, hp["pak_ep"], hp["opt"], lr_b, lr_h, lin, tag=f"{name}-PAK-{mode}")
    res = evaluate(model, PAK_LOADERS["test"], EXPECT_PAK)
    log_result(name, "stage2", mode, "PAK test", res, model, mins)
    HISTORIES[f"{name}/{mode}"] = hist
    PREDS[(name, mode)] = (res["targets"], res["preds"])
    hist.to_csv(OUT_DIR / f"hist_{name}_{mode}.csv", index=False)           # saved so resumed sessions can still plot
    np.savez(OUT_DIR / f"preds_{name}_{mode}.npz", t=res["targets"], p=res["preds"])
    torch.save(model.state_dict(), OUT_DIR / f"{name}_pak_{mode}.pt")
    del model; torch.cuda.empty_cache()

DONE = {(r["model"], r["stage"], r["setting"]) for r in RESULTS}
for name in ["zfnet", "cvt"]:
    for mode in ["baseline", "us_finetune", "us_linear"]:
        if (name, "stage2", mode) in DONE:
            print(f"[{name}-PAK-{mode}] already done, skipping")
            load_saved_hist(f"{name}/{mode}", OUT_DIR / f"hist_{name}_{mode}.csv")
            f = OUT_DIR / f"preds_{name}_{mode}.npz"
            if f.exists():
                z = np.load(f); PREDS[(name, mode)] = (z["t"], z["p"])
            continue
        run_pak(name, mode)

## 6. Results

In [ ]:
df = pd.DataFrame(RESULTS)
display(df)
df.to_csv(OUT_DIR / "results.csv", index=False)

pak = df[df.dataset == "PAK test"]
ax = pak.pivot(index="setting", columns="model", values="top1").plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel("Top-1 accuracy (%) on Pakistani test split"); ax.set_title("Stage 2 comparison")
plt.tight_layout(); plt.savefig(OUT_DIR / "pak_comparison.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, h in HISTORIES.items():
    axes[0].plot(h.epoch, h.val_acc, label=key); axes[1].plot(h.epoch, h.train_loss, label=key)
axes[0].set_title("Validation accuracy"); axes[1].set_title("Training loss")
for a in axes: a.set_xlabel("epoch")
axes[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(OUT_DIR / "curves.png", dpi=200); plt.show()

# confusion matrix + per-class accuracy for the best Pakistani run
have = pak[[(m, s) in PREDS for m, s in zip(pak.model, pak.setting)]]   # runs whose predictions are available
assert len(have) > 0, "no saved predictions for any Stage 2 run"
best = have.sort_values("top1", ascending=False).iloc[0]
print("confusion matrix for:", best.model, best.setting)
t, p = PREDS[(best.model, best.setting)]
cm = confusion_matrix(t, p, labels=range(EXPECT_PAK), normalize="true")
plt.figure(figsize=(12, 10)); sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
plt.title(f"Normalised confusion matrix: {best.model} / {best.setting} (Pakistani test)")
plt.xlabel("predicted"); plt.ylabel("true"); plt.tight_layout()
plt.savefig(OUT_DIR / "pak_confusion.png", dpi=200); plt.show()

per_class = pd.DataFrame({"class": PAK_CLASSES, "accuracy": cm.diagonal()}).sort_values("accuracy")
per_class.to_csv(OUT_DIR / "pak_per_class_accuracy.csv", index=False)
print("Hardest classes:\n", per_class.head(8).to_string(index=False))

In [ ]:
t, p = PREDS[(best.model, best.setting)]
names = np.array(PAK_CLASSES)
print("run:", best.model, best.setting)
weak = per_class.head(8)["class"].tolist()
for c in weak:
    ci = PAK_CLASSES.index(c)
    mask = t == ci
    wrong = pd.Series(names[p[mask][p[mask] != ci]]).value_counts().head(3)
    print(f"{c}: {mask.sum()} test images, mistaken for -> {dict(wrong)}")

## Notes for the report
- Report the number of parameters from section 2 and the `params_M` / `trainable_M` columns in the results table.
- ZFNet had no ImageNet pretraining, so its US-trained backbone is the only "pretrained" source. Say this explicitly.
- `baseline` vs `us_finetune` shows the value of US car pretraining. `us_linear` vs `us_finetune` shows how transferable the frozen features are.
- Splits: the provided train/ and test/ folders were merged and re-split 70/10/20 per dataset (stratified, seed 42). State this in the report.
- All numbers are from a single seed. Re-run with `SEED` changed if you want mean and standard deviation.